# 01 - Data Collection and Understanding

**Objective:** Load the Medical Cost Personal Dataset, inspect it carefully, and decide whether it is safe to model. We do **not** assume the data is clean - we check.

**Role in the ML workflow:** *Pandas and EDA* - inspect, clean, transform and understand the data. Every later stage depends on getting this right.

> **SQL reminder:** In a real company the data would usually live in a database. SQL would *extract, filter, join and aggregate* the rows, and pandas would take over from there. At the end of this notebook there is a short note on how this project could use SQL in a future version.

### Concepts to understand first
* **Supervised regression:** we have inputs (features) *and* a known numeric answer (the target). The model learns the relationship so it can estimate the answer for new people.
* **DataFrame:** pandas' table. Rows = observations (people), columns = variables.
* **Numerical vs categorical variable:** numerical = measured quantities where arithmetic makes sense (age, BMI). Categorical = labels (sex, region). Models need categorical variables converted to numbers (Stage 3).

## Step 1 - Import libraries

In [1]:
import sys
import warnings
from pathlib import Path

# Make the project root importable so we can use the code in src/.
# (Jupyter's working directory is notebooks/, so the project root is one level up.)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from src.data_preprocessing import (
    DATA_URL, RAW_DATA_PATH, TARGET, FEATURE_COLUMNS,
    NUMERICAL_COLUMNS, CATEGORICAL_COLUMNS,
)

pd.set_option("display.max_columns", None)  # show every column when printing a DataFrame
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.3


*What this does:* `pandas` handles tables, `numpy` handles numbers. The constants come from `src/data_preprocessing.py` so every notebook and script agrees on column names and the file location (one source of truth, no copy-pasting).

## Step 2 - Load the dataset

The dataset is the *Medical Cost Personal Dataset* hosted at:

`https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv`

The cell below uses the local copy in `data/raw/`. If that file is missing it downloads it from the URL once and saves it, so the project also works on a fresh clone.

In [3]:
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.read_csv(DATA_URL).to_csv(RAW_DATA_PATH, index=False)   # one-time download
    print("Downloaded dataset to", RAW_DATA_PATH)

df = pd.read_csv(RAW_DATA_PATH)
print("Loaded from:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))

Loaded from: data/raw/insurance.csv


*Why:* keeping a raw copy means the analysis is reproducible even if the website changes. The raw file is **never edited** - all cleaning happens on copies in code.

## Step 3 - First and last five rows

In [4]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [5]:
df.tail()

,age,sex,bmi,children,smoker,region,charges
1333,50,male,30.97,3,no,northwest,10600.5483
1334,18,female,31.92,0,no,northeast,2205.9808
1335,18,female,36.85,0,no,southeast,1629.8335
1336,21,female,25.80,0,no,southwest,2007.9450
1337,61,female,29.07,0,yes,northwest,29141.3603


*Why look at both ends?* A file can be sorted or have junk (footers, totals) at the end. Seeing both ends is a cheap sanity check. Here each row is one insured person.

## Step 4 - Dimensions

In [6]:
n_rows, n_cols = df.shape
print(f"Rows (people): {n_rows}")
print(f"Columns      : {n_cols}")

Rows (people): 1338
Columns      : 7


## Step 5 - Column names and data types

In [7]:
print(list(df.columns))
print()
print(df.dtypes)

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


*Why:* a number stored as text (e.g. `"27.9"`) or a category stored as a number would silently break later steps. Expected: `age`, `children` are integers, `bmi` and `charges` are floats, and `sex`, `smoker`, `region` are text (displayed as `object` or `str` depending on your pandas version).

## Step 6 - `info()` and `describe()`

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [9]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


* `info()` shows non-null counts per column (missing values appear as counts below the number of rows) and memory use.
* `describe()` summarises **numeric** columns only: count, mean, std, min, quartiles, max. We look for impossible values (negative ages, zero BMI) and for a mean far from the median (skew).

In [10]:
df.select_dtypes(exclude="number").describe()   # text columns only

,sex,smoker,region
count,1338,1338,1338
unique,2,2,4
top,male,no,southeast
freq,676,1064,364


*Categorical summary:* number of distinct values and the most frequent one for each text column.

## Step 7 - Missing values and duplicate records

In [11]:
print("Missing values per column:")
print(df.isna().sum())
print("\nTotal missing:", int(df.isna().sum().sum()))
print("Duplicated rows:", int(df.duplicated().sum()))

Missing values per column:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Total missing: 0
Duplicated rows: 1


In [12]:
# Show every row that belongs to a duplicate group (keep=False marks all copies)
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


**Reading the result:** there are no missing values, but there **is one duplicated record** (two rows identical in all seven columns, including `charges` to four decimals). We *only inspect* it here; the decision on what to do with it is made and justified in notebook 02.

## Step 8 - Numerical and categorical variables

| Column | Meaning | Type |
|---|---|---|
| `age` | Age of the person (years) | Numerical (discrete) |
| `sex` | `female` / `male` | Categorical (nominal, 2 levels) |
| `bmi` | Body Mass Index = weight (kg) / height (m)². Roughly: <18.5 underweight, 18.5-25 normal, 25-30 overweight, >=30 obese | Numerical (continuous) |
| `children` | Number of dependants covered by the insurance | Numerical (discrete count) |
| `smoker` | `yes` / `no` | Categorical (binary) |
| `region` | US residential region: northeast, northwest, southeast, southwest | Categorical (nominal, 4 levels) |
| `charges` | Individual medical insurance charges (currency units) | Numerical (continuous) - **the target** |

In [13]:
print("Numerical features  :", NUMERICAL_COLUMNS)
print("Categorical features :", CATEGORICAL_COLUMNS)
print("Target               :", TARGET)

# Verify the lists above against what pandas actually found
auto_numeric = df.drop(columns=TARGET).select_dtypes(include="number").columns.tolist()
print("\npandas sees as numeric:", auto_numeric)
assert sorted(auto_numeric) == sorted(NUMERICAL_COLUMNS), "numeric columns differ from our list!"

Numerical features  : ['age', 'bmi', 'children']
Categorical features : ['sex', 'smoker', 'region']
Target               : charges

pandas sees as numeric: ['age', 'bmi', 'children']


`children` is a *count*, so treating it as a number is valid (2 children is "more" than 1). In notebook 03 we test that choice against one-hot encoding rather than just assuming it.

## Step 9 - Invalid or unusual values

In [14]:
# 1) Do the categorical columns contain only the expected labels (no typos, no 'Male ' vs 'male')?
for col in CATEGORICAL_COLUMNS:
    print(col, "->", df[col].value_counts().to_dict())

sex -> {'male': 676, 'female': 662}
smoker -> {'no': 1064, 'yes': 274}
region -> {'southeast': 364, 'southwest': 325, 'northwest': 325, 'northeast': 324}


In [15]:
# 2) Range checks on the numeric columns
checks = {
    "age <= 0":            (df["age"] <= 0).sum(),
    "age > 100":           (df["age"] > 100).sum(),
    "bmi <= 0":            (df["bmi"] <= 0).sum(),
    "children < 0":        (df["children"] < 0).sum(),
    "charges <= 0":        (df[TARGET] <= 0).sum(),
    "non-integer age":     (df["age"] % 1 != 0).sum(),
    "non-integer children":(df["children"] % 1 != 0).sum(),
}
print(pd.Series(checks, name="rows violating rule"))
print()
print("age range     :", df["age"].min(), "to", df["age"].max())
print("bmi range     :", df["bmi"].min(), "to", df["bmi"].max())
print("children range:", df["children"].min(), "to", df["children"].max())
print("charges range :", round(df[TARGET].min(), 2), "to", round(df[TARGET].max(), 2))

age <= 0                0
age > 100               0
bmi <= 0                0
children < 0            0
charges <= 0            0
non-integer age         0
non-integer children    0
Name: rows violating rule, dtype: int64

age range     : 18 to 64
bmi range     : 15.96 to 53.13
children range: 0 to 5
charges range : 1121.87 to 63770.43


**Reading the result:** no rule is violated. Two things are *unusual but plausible* and must **not** be deleted blindly:
* very high BMI values (up to 53) - severe obesity exists;
* very high charges (up to about 64,000) - expensive medical cases are exactly what an insurance model must learn about.

Also note: all ages are between 18 and 64, so the model will only be trustworthy for adults in that range.

## Step 10 - The prediction problem

* **Problem type:** supervised **regression** (the answer is a continuous number).
* **Target (`y`):** `charges` - the insurance charge for one person.
* **Features (`X`):** `age`, `sex`, `bmi`, `children`, `smoker`, `region`.
* **Goal:** learn a function `charges ~ f(age, sex, bmi, children, smoker, region)` that generalises to people the model has not seen.
* **Important caution:** this dataset is a small public sample. A model trained on it gives *estimates for learning purposes*, not real insurance prices.

In [16]:
X = df[FEATURE_COLUMNS]
y = df[TARGET]
print("X (features) shape:", X.shape)
print("y (target)   shape:", y.shape)

X (features) shape: (1338, 6)
y (target)   shape: (1338,)


## Summary of Stage One

| Check | Result |
|---|---|
| Shape | 1,338 rows x 7 columns |
| Missing values | none |
| Duplicates | 1 duplicated row (handled in notebook 02) |
| Invalid values | none found; extreme BMI / charges look legitimate |
| Target | `charges` (continuous) |

### Common errors and fixes
| Error | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'src'` | notebook's working directory is wrong | open the project **folder** in VS Code (File > Open Folder) and re-run the setup cell |
| `FileNotFoundError` for the CSV | file is not in `data/raw/` | place `insurance.csv` there, or let the download cell fetch it |
| `ModuleNotFoundError: pandas` | wrong kernel selected | choose the kernel of your virtual environment (top-right of the notebook) |

### Practical task
1. Write one line of pandas that counts how many people are `smoker == "yes"`.
2. Find the age of the oldest person and the highest `charges` value.
3. In your own words, explain why `region` cannot be fed directly into a regression model.

### Future version: SQL
If the data were in a database table `insurance`, you would replace the CSV load with a query, e.g.
```sql
SELECT age, sex, bmi, children, smoker, region, charges
FROM insurance
WHERE age BETWEEN 18 AND 64;
```
and read it with `pd.read_sql(query, connection)`. SQL would also do `GROUP BY region` aggregations or `JOIN` a separate `policies` table before the data reaches pandas. The rest of the pipeline would stay unchanged.

**Next:** notebook 02 - data cleaning and exploratory data analysis.